# Benchmark 3W — Spark vs Flink (sem Kafka/Docker)

Carrega os arquivos parquet do dataset **Petrobras 3W**, processa com **Spark Structured Streaming** e **PyFlink**, e plota métricas de **latência** e **throughput**.

**Suporte a GPU:**
- `cuDF` → leitura de parquet na GPU
- `CuPy` → agregação de janelas nos operadores Flink
- RAPIDS Accelerator JAR → SQL/shuffle do Spark na GPU

---
| Variável | Descrição |
|---|---|
| `DATASET_PATH` | Caminho para o diretório raiz do 3W |
| `MAX_FILES_PER_CLASS` | Arquivos parquet por classe de evento |
| `MAX_ROWS` | Linhas a processar (0 = sem limite) |
| `USE_GPU` | `True` = GPU, `False` = CPU, `None` = auto |
| `ENGINE` | `"spark"`, `"flink"` ou `"both"` |

## ⚙️ Setup Colab

Execute esta célula **uma vez** antes de tudo.
Escolha **uma** das opções de dataset:

| Opção | Quando usar |
|---|---|
| **A — GitHub** | Primeira vez, dataset público (~1 GB com LFS) |
| **B — Google Drive** | Já fez upload; mais rápido nas próximas sessões |

> **Colab Pro + GPU:** vá em *Ambiente de execução → Alterar tipo → GPU T4/A100*

In [ ]:
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

# ------------------------------------------------------------------
# ESCOLHA UMA OPÇÃO:
# ------------------------------------------------------------------
DATASET_SOURCE = "github"   # "github" | "drive"
# ------------------------------------------------------------------

if IN_COLAB:
    # --- Dependências ---
    print("Instalando dependências...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "apache-flink>=1.18.0",
        "pyspark>=3.5.0,<4.0",
        "pandas>=2.1.0",
        "pyarrow>=14.0.0",
        "numpy>=1.24.0",
        "matplotlib>=3.8.0",
        "seaborn>=0.13.0",
    ], check=True)

    if DATASET_SOURCE == "github":
        # --- Opção A: clonar do GitHub (Git LFS) ---
        if not os.path.exists("/content/3W/dataset/0"):
            print("Clonando dataset 3W do GitHub (pode levar alguns minutos)...")
            subprocess.run(["apt-get", "install", "-qq", "git-lfs"], check=True)
            subprocess.run(["git", "lfs", "install"], check=True)
            subprocess.run(
                ["git", "clone", "--depth=1",
                 "https://github.com/petrobras/3W.git",
                 "/content/3W"],
                check=True,
            )
        DATASET_PATH = "/content/3W/dataset"

    elif DATASET_SOURCE == "drive":
        # --- Opção B: montar Google Drive ---
        # Estrutura esperada no Drive:
        #   Meu Drive/
        #     3W/
        #       dataset/
        #         0/  1/  2/ ... (diretórios por classe de evento)
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DATASET_PATH = "/content/drive/MyDrive/3W/dataset"

    OUTPUT_DIR_STR = "/content/results"
    PLOTS_DIR_STR  = "/content/plots"
    print(f"Dataset path: {DATASET_PATH}")
    print(f"GPU disponível: {os.path.exists('/proc/driver/nvidia/gpus')}")
else:
    # Ambiente local — usa os defaults da célula de configuração abaixo
    DATASET_PATH   = None   # será sobrescrito pela célula de configuração
    OUTPUT_DIR_STR = None
    PLOTS_DIR_STR  = None
    print("Ambiente local detectado — usando configurações da próxima célula")

## 0. Configuração

In [ ]:
import os
from pathlib import Path

DATASET_PATH       = os.environ.get("DATASET_PATH", "/content/jonathan/3W/dataset")
MAX_FILES_PER_CLASS = 5        # arquivos parquet por classe de evento (0-9)
MAX_ROWS           = 100_000   # 0 = sem limite
USE_GPU            = None      # None = auto-detecta | True = força GPU | False = força CPU
ENGINE             = "both"    # "spark" | "flink" | "both"

OUTPUT_DIR = Path("results")
PLOTS_DIR  = Path("plots")
OUTPUT_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

# Caminho do JAR RAPIDS para Spark (opcional)
# Baixe com: wget <url> -O benchmark_simple/rapids-4-spark.jar
RAPIDS_JAR_PATH = os.environ.get("RAPIDS_JAR_PATH", str(Path("rapids-4-spark.jar")))

## 1. Imports e detecção de GPU

In [ ]:
import json
import shutil
import subprocess
import tempfile
import time
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
    print("seaborn:", sns.__version__)
except ImportError:
    print("seaborn não instalado — usando matplotlib padrão")

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

In [ ]:
def detect_gpu() -> bool:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if r.returncode == 0 and r.stdout.strip():
            for line in r.stdout.strip().splitlines():
                print(f"  GPU detectada: {line.strip()}")
            return True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return False

def check_cudf() -> bool:
    try:
        import cudf  # noqa: F401
        return True
    except ImportError:
        return False

def check_cupy() -> bool:
    try:
        import cupy as cp
        cp.array([1.0])
        return True
    except (ImportError, Exception):
        return False

# Resolve USE_GPU
if USE_GPU is None:
    USE_GPU = detect_gpu()

print(f"\nModo: {'GPU' if USE_GPU else 'CPU'}")
print(f"cuDF disponível : {check_cudf()}")
print(f"CuPy disponível : {check_cupy()}")

## 2. Colunas de sensores

In [ ]:
SENSOR_COLS = [
    "P-PDG", "P-TPT", "T-TPT", "P-MON-CKP", "T-JUS-CKP",
    "P-JUS-CKP", "P-MON-CKGL", "P-JUS-CKGL", "QGL", "QBS",
]

## 3. Carregamento do dataset 3W

In [ ]:
def load_3w_dataset(
    dataset_path: str,
    max_files_per_class: int = 5,
    max_rows: int = 100_000,
    use_gpu: bool = False,
) -> pd.DataFrame:
    """
    Lê arquivos parquet do dataset 3W e retorna um DataFrame pandas consolidado.
    Se use_gpu=True e cuDF estiver disponível, tenta leitura na GPU com fallback
    automático para pandas quando o cuDF falhar (ex: parquets com compressão
    não suportada pela versão do RAPIDS do Colab).
    """
    dataset_path = Path(dataset_path)
    dfs = []
    total_files = 0

    use_cudf = use_gpu and check_cudf()
    if use_cudf:
        import cudf as _cudf
        print("[cuDF] Lendo parquets na GPU (fallback automático para pandas)")
    elif use_gpu:
        print("[CPU] cuDF não disponível — usando pandas")

    def _read_parquet(pf):
        """Lê um parquet, tentando cuDF primeiro e caindo para pandas se falhar."""
        if use_cudf:
            try:
                gdf = _cudf.read_parquet(str(pf))
                gdf.index.name = "timestamp"
                return gdf.reset_index().to_pandas(), "gpu"
            except Exception as e:
                print(f"  [WARN cuDF→pandas] {pf.name}: {e}")
        df = pd.read_parquet(pf)
        df.index.name = "timestamp"
        return df.reset_index(), "cpu"

    for event_dir in sorted(dataset_path.iterdir()):
        if not event_dir.is_dir() or not event_dir.name.isdigit():
            continue
        event_code = int(event_dir.name)
        for pf in sorted(event_dir.glob("*.parquet"))[:max_files_per_class]:
            try:
                df, backend = _read_parquet(pf)
                stem   = pf.stem
                source = (
                    "REAL"      if stem.startswith("WELL-") else
                    "SIMULATED" if stem.startswith("SIMULATED") else "DRAWN"
                )
                df["well_id"]      = stem
                df["source"]       = source
                df["event_code"]   = event_code
                df["timestamp_ms"] = (
                    pd.to_datetime(df["timestamp"]).astype("int64") // 1_000_000
                )
                for col in SENSOR_COLS:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                dfs.append(df)
                total_files += 1
            except Exception as exc:
                print(f"  [SKIP] {pf.name}: {exc}")

    if not dfs:
        raise RuntimeError(f"Nenhum parquet encontrado em {dataset_path}")

    combined = pd.concat(dfs, ignore_index=True).sort_values("timestamp_ms").reset_index(drop=True)
    total_loaded = len(combined)

    if max_rows > 0 and total_loaded > max_rows:
        combined = (
            combined.sample(n=max_rows, random_state=42)
            .sort_values("timestamp_ms")
            .reset_index(drop=True)
        )
        print(f"Dataset: {total_files} arquivos | {len(combined):,} linhas (de {total_loaded:,})")
    else:
        print(f"Dataset: {total_files} arquivos | {len(combined):,} linhas")

    return combined


In [ ]:
df = load_3w_dataset(
    DATASET_PATH,
    max_files_per_class=MAX_FILES_PER_CLASS,
    max_rows=MAX_ROWS,
    use_gpu=USE_GPU,
)

print(f"\nColunas de sensores disponíveis: {[c for c in SENSOR_COLS if c in df.columns]}")
print(f"Wells únicos: {df['well_id'].nunique()}")
print(f"Classes de evento: {sorted(df['event_code'].unique())}")
df[["well_id", "event_code", "source", "timestamp_ms"] + [c for c in SENSOR_COLS[:3] if c in df.columns]].head()

## 4. Benchmark Spark Structured Streaming

**Pipeline:**
1. Escreve o DataFrame em chunks parquet num diretório temporário (simula fonte de streaming)
2. `readStream.format("parquet")` + `trigger(once=True)` processa todos os dados de uma vez
3. Watermark + janela de 60s com `mean` e `stddev` por `well_id`
4. Coleta latência por janela: `processing_ts_ms − ingestion_ts_ms`

**GPU:** RAPIDS Accelerator JAR redireciona SQL/shuffle/join para a GPU sem mudanças no código.

In [ ]:
def run_spark_benchmark(df: pd.DataFrame, use_gpu: bool = False) -> dict[str, Any]:
    try:
        from pyspark.sql import SparkSession
        import pyspark.sql.functions as F
        import pyspark
    except ImportError:
        print("[ERRO] PySpark não encontrado.")
        return {}

    tmpdir = tempfile.mkdtemp(prefix="3w_spark_")
    try:
        builder = (
            SparkSession.builder
            .appName("3W-Benchmark-Spark" + ("-GPU" if use_gpu else ""))
            .master("local[*]")
            .config("spark.sql.shuffle.partitions", "8")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "2g")
            .config("spark.ui.enabled", "false")
            .config("spark.sql.streaming.checkpointLocation", f"{tmpdir}/checkpoint")
        )

        # RAPIDS só suporta Spark 3.3–3.5; desativa automaticamente no 4.x
        _spark_major = int(pyspark.__version__.split(".")[0])
        _rapids_supported = _spark_major < 4

        if use_gpu and not _rapids_supported:
            print(f"[AVISO] RAPIDS não suporta Spark {pyspark.__version__} — rodando GPU via CuPy/CPU only")
            use_gpu = False  # desativa RAPIDS, mantém CuPy no Flink

        if use_gpu:
            rapids_jar = Path(RAPIDS_JAR_PATH)
            if rapids_jar.exists():
                print(f"[GPU] RAPIDS JAR: {rapids_jar}")
                builder = (
                    builder
                    .config("spark.jars", str(rapids_jar))
                    .config("spark.plugins", "com.nvidia.spark.SQLPlugin")
                    .config("spark.rapids.sql.enabled", "true")
                    .config("spark.rapids.sql.incompatibleOps.enabled", "true")
                    .config("spark.rapids.memory.pinnedPool.size", "2g")
                    .config("spark.executor.resource.gpu.amount", "1")
                    .config("spark.task.resource.gpu.amount", "0.25")
                    .config("spark.rapids.sql.concurrentGpuTasks", "4")
                )
            else:
                spark_ver = pyspark.__version__
                # Escolhe a versão do JAR compatível com Spark 3.5.x
                coord = "com.nvidia:rapids-4-spark_2.12:24.10.0"
                print(f"[GPU] JAR não encontrado — baixando via Maven: {coord}")
                builder = (
                    builder
                    .config("spark.jars.packages", coord)
                    .config("spark.plugins", "com.nvidia.spark.SQLPlugin")
                    .config("spark.rapids.sql.enabled", "true")
                    .config("spark.executor.resource.gpu.amount", "1")
                    .config("spark.task.resource.gpu.amount", "0.25")
                )

        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("ERROR")

        # Prepara fonte de streaming
        ingestion_ts_ms = int(time.time() * 1000)
        df_copy = df.copy()
        df_copy["producer_ts_ms"] = ingestion_ts_ms

        n_chunks = min(20, max(1, len(df_copy) // 5000))
        chunk_size = max(1, len(df_copy) // n_chunks)
        for i in range(0, len(df_copy), chunk_size):
            df_copy.iloc[i : i + chunk_size].to_parquet(f"{tmpdir}/chunk_{i:08d}.parquet", index=False)

        schema = spark.read.parquet(tmpdir).schema
        available = [c for c in SENSOR_COLS if c in df.columns]
        agg_exprs = (
            [F.avg(c).alias(f"avg_{c.lower().replace('-','_')}") for c in available]
            + [F.stddev(c).alias(f"std_{c.lower().replace('-','_')}") for c in available]
        )

        sdf = spark.readStream.format("parquet").schema(schema).load(tmpdir)
        result_sdf = (
            sdf
            .withColumn("event_time", (F.col("timestamp_ms") / 1000).cast("timestamp"))
            .withWatermark("event_time", "10 seconds")
            .groupBy(F.window("event_time", "60 seconds"), "well_id", "event_code")
            .agg(
                F.count("*").alias("record_count"),
                F.max("producer_ts_ms").alias("max_producer_ts_ms"),
                *agg_exprs,
            )
            .withColumn("processing_ts_ms", (F.unix_timestamp() * 1000).cast("long"))
            .withColumn("latency_ms", F.col("processing_ts_ms") - F.col("max_producer_ts_ms"))
        )

        t_start = time.time()
        query = (
            result_sdf.writeStream
            .format("memory").queryName("spark_results")
            .trigger(once=True).start()
        )
        query.awaitTermination(timeout=600)
        t_end = time.time()
        total_time_s = t_end - t_start

        progress = query.lastProgress or {}
        num_input_rows = progress.get("numInputRows", len(df))
        duration_ms = progress.get("durationMs", {})

        rows = spark.sql(
            "SELECT latency_ms, record_count FROM spark_results WHERE latency_ms IS NOT NULL"
        ).collect()
        latencies   = [r["latency_ms"]   for r in rows if r["latency_ms"]   is not None]
        win_sizes   = [r["record_count"] for r in rows if r["record_count"] is not None]
        throughput  = num_input_rows / total_time_s if total_time_s > 0 else 0.0

        metrics = {
            "engine": "spark", "gpu_enabled": use_gpu,
            "total_time_s": round(total_time_s, 3),
            "total_records": int(len(df)),
            "windows_processed": len(latencies),
            "throughput_rps": round(throughput, 2),
            "input_rows_per_second": progress.get("inputRowsPerSecond", throughput),
            "processed_rows_per_second": progress.get("processedRowsPerSecond", throughput),
            "trigger_execution_ms": duration_ms.get("triggerExecution", 0),
            "latency_ms_raw": latencies, "window_sizes": win_sizes,
            "latency_p50": float(np.percentile(latencies, 50)) if latencies else 0.0,
            "latency_p95": float(np.percentile(latencies, 95)) if latencies else 0.0,
            "latency_p99": float(np.percentile(latencies, 99)) if latencies else 0.0,
            "latency_mean": float(np.mean(latencies)) if latencies else 0.0,
        }

        tag = "GPU" if use_gpu else "CPU"
        print(f"[Spark/{tag}] Tempo: {total_time_s:.2f}s | Throughput: {throughput:.0f} rec/s | "
              f"Janelas: {len(latencies)} | p50/p95/p99: "
              f"{metrics['latency_p50']:.0f}/{metrics['latency_p95']:.0f}/{metrics['latency_p99']:.0f} ms")

        spark.stop()
        return metrics
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

## 5. Benchmark Flink (PyFlink + CuPy)

**Pipeline:**
1. `env.from_collection()` cria um DataStream finito a partir do DataFrame
2. `_GPUWindowAggregator` acumula 60 registros por `well_id` e calcula `mean`/`std` de cada sensor
3. `execute_and_collect()` executa o job e retorna todos os resultados

**GPU:** `CuPy` substitui `NumPy` para os cálculos vetoriais dentro de cada janela.

In [ ]:
class _GPUWindowAggregator:
    """
    Acumulador de janela por well_id usando CuPy (GPU) ou NumPy (CPU).
    Emite resultado a cada `window_size` registros por well.
    """
    def __init__(self, n_sensors: int, window_size: int = 60, use_gpu: bool = False):
        self._n          = n_sensors
        self._window_size = window_size
        self._use_gpu    = use_gpu and check_cupy()
        self._buffers: dict = {}
        self._xp = __import__("cupy") if self._use_gpu else np

    def _compute(self, data):
        xp = self._xp
        arr = xp.array(data, dtype=xp.float32)
        if self._use_gpu:
            import cupy as cp
            valid  = cp.where(cp.isnan(arr), 0.0, arr)
            count  = cp.sum(~cp.isnan(arr), axis=0).clip(1)
            means  = (cp.sum(valid, axis=0) / count).tolist()
            diff   = valid - cp.sum(valid, axis=0, keepdims=True) / count
            stds   = cp.sqrt(cp.sum(diff**2, axis=0) / count).tolist()
        else:
            means = np.nanmean(arr, axis=0).tolist()
            stds  = np.nanstd(arr,  axis=0).tolist()
        return means, stds

    def push(self, ingest_ts, well_id, ts_ms, event_code, sensors):
        buf = self._buffers.setdefault(well_id, {"data": [], "first_ingest": ingest_ts})
        buf["data"].append(sensors)
        if len(buf["data"]) >= self._window_size:
            means, stds = self._compute(buf["data"])
            first_ingest = buf["first_ingest"]
            self._buffers[well_id] = {"data": [], "first_ingest": int(time.time() * 1000)}
            proc_ts = int(time.time() * 1000)
            return (well_id, ts_ms, event_code, proc_ts - first_ingest, proc_ts, *means, *stds)
        return None

    def flush_all(self):
        results = []
        for well_id, buf in self._buffers.items():
            if buf["data"]:
                means, stds = self._compute(buf["data"])
                proc_ts = int(time.time() * 1000)
                results.append((well_id, 0, 0, proc_ts - buf["first_ingest"], proc_ts, *means, *stds))
        return results

In [ ]:
def run_flink_benchmark(df: pd.DataFrame, use_gpu: bool = False) -> dict[str, Any]:
    try:
        from pyflink.datastream import StreamExecutionEnvironment
        from pyflink.common.typeinfo import Types
        from pyflink.datastream.functions import MapFunction
    except ImportError:
        print("[ERRO] PyFlink não encontrado. Instale: pip install apache-flink")
        return {}

    use_cupy = use_gpu and check_cupy()
    tag = "GPU/CuPy" if use_cupy else "CPU/NumPy"
    if use_gpu and not use_cupy:
        print("[WARN] CuPy não disponível — Flink rodará em CPU")

    available = [c for c in SENSOR_COLS if c in df.columns]
    n_sensors = len(available)
    n_stats   = n_sensors * 2

    field_types = (
        [Types.LONG(), Types.STRING(), Types.LONG(), Types.INT()]
        + [Types.DOUBLE()] * n_sensors
    )
    output_type = Types.TUPLE(
        [Types.STRING(), Types.LONG(), Types.INT(), Types.LONG(), Types.LONG()]
        + [Types.DOUBLE()] * n_stats
    )

    class WindowAggMap(MapFunction):
        def __init__(self, n, gpu):
            self._n = n; self._gpu = gpu; self._agg = None
        def open(self, ctx):
            self._agg = _GPUWindowAggregator(self._n, window_size=60, use_gpu=self._gpu)
        def map(self, value):
            ingest_ts, well_id, ts_ms, event_code = value[0], value[1], value[2], value[3]
            sensors = [value[4 + i] for i in range(self._n)]
            result = self._agg.push(ingest_ts, well_id, ts_ms, event_code, sensors)
            if result:
                return result
            proc_ts = int(time.time() * 1000)
            return (well_id, ts_ms, event_code, proc_ts - ingest_ts, proc_ts, *[0.0] * n_stats)

    print("[Flink] Inicializando ambiente...")
    env = StreamExecutionEnvironment.get_execution_environment()
    env.set_parallelism(1)

    ingest_ts_ms = int(time.time() * 1000)
    print(f"[Flink] Preparando {len(df):,} registros ({tag})...")

    col_data    = {c: df[c].fillna(0.0).astype(float).tolist() for c in available}
    well_ids    = df["well_id"].astype(str).tolist()
    timestamps  = df["timestamp_ms"].fillna(0).astype(int).tolist()
    event_codes = df["event_code"].fillna(0).astype(int).tolist()

    records = [
        (ingest_ts_ms, well_ids[i], timestamps[i], event_codes[i],
         *[col_data[c][i] for c in available])
        for i in range(len(df))
    ]

    ds = env.from_collection(
        records,
        type_info=Types.TUPLE(field_types),
    )
    result_ds = ds.map(WindowAggMap(n_sensors, use_cupy), output_type=output_type)

    print(f"[Flink] Executando job ({tag})...")
    t_start = time.time()
    collected = list(result_ds.execute_and_collect())
    t_end = time.time()

    total_time_s = t_end - t_start
    throughput   = len(df) / total_time_s if total_time_s > 0 else 0.0

    window_latencies = [
        int(item[3]) for item in collected
        if any(item[5 + n_sensors + k] != 0.0 for k in range(n_sensors))
    ]
    latencies = window_latencies or [int(item[3]) for item in collected][:5000]

    metrics = {
        "engine": "flink", "gpu_enabled": use_gpu, "cupy_used": use_cupy,
        "total_time_s": round(total_time_s, 3),
        "total_records": len(records), "records_collected": len(collected),
        "windows_emitted": len(window_latencies),
        "throughput_rps": round(throughput, 2),
        "latency_ms_raw": latencies,
        "latency_p50":  float(np.percentile(latencies, 50))  if latencies else 0.0,
        "latency_p95":  float(np.percentile(latencies, 95))  if latencies else 0.0,
        "latency_p99":  float(np.percentile(latencies, 99))  if latencies else 0.0,
        "latency_mean": float(np.mean(latencies))            if latencies else 0.0,
    }

    print(f"[Flink/{tag}] Tempo: {total_time_s:.2f}s | Throughput: {throughput:.0f} rec/s | "
          f"Janelas: {len(window_latencies)} | p50/p95/p99: "
          f"{metrics['latency_p50']:.0f}/{metrics['latency_p95']:.0f}/{metrics['latency_p99']:.0f} ms")

    return metrics

## 6. Execução

In [ ]:
results: dict[str, dict] = {}

if ENGINE in ("spark", "both"):
    print("=" * 55)
    print(f"  Spark {'(GPU/RAPIDS)' if USE_GPU else '(CPU)'}")
    print("=" * 55)
    spark_metrics = run_spark_benchmark(df, use_gpu=USE_GPU)
    if spark_metrics:
        results["spark"] = spark_metrics

if ENGINE in ("flink", "both"):
    print("=" * 55)
    print(f"  Flink {'(GPU/CuPy)' if USE_GPU else '(CPU/NumPy)'}")
    print("=" * 55)
    flink_metrics = run_flink_benchmark(df, use_gpu=USE_GPU)
    if flink_metrics:
        results["flink"] = flink_metrics

## 7. Resumo e persistência

In [ ]:
# Salva JSONs
for engine, m in results.items():
    out = {k: v for k, v in m.items() if k != "latency_ms_raw"}
    out["latency_ms_sample"] = m.get("latency_ms_raw", [])[:2000]
    path = OUTPUT_DIR / f"{engine}_metrics.json"
    path.write_text(json.dumps(out, indent=2))
    print(f"Salvo: {path}")

if results:
    summary = {}
    for engine, m in results.items():
        summary[engine] = {k: v for k, v in m.items() if k != "latency_ms_raw"}
        summary[engine]["latency_ms_sample"] = m.get("latency_ms_raw", [])[:2000]
    (OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
    print(f"Salvo: {OUTPUT_DIR / 'summary.json'}")

In [ ]:
# Tabela resumo
if results:
    engines = list(results.keys())
    rows = []
    for key, label in [
        ("gpu_enabled",    "GPU ativado"),
        ("total_time_s",   "Tempo total (s)"),
        ("total_records",  "Total registros"),
        ("throughput_rps", "Throughput (rec/s)"),
        ("latency_p50",    "Latência P50 (ms)"),
        ("latency_p95",    "Latência P95 (ms)"),
        ("latency_p99",    "Latência P99 (ms)"),
        ("latency_mean",   "Latência média (ms)"),
    ]:
        row = {"Métrica": label}
        for e in engines:
            v = results[e].get(key, "N/A")
            row[e.capitalize()] = f"{v:,.2f}" if isinstance(v, float) else (f"{v:,}" if isinstance(v, int) else str(v))
        rows.append(row)
    pd.DataFrame(rows).set_index("Métrica")

## 8. Plots

In [ ]:
from IPython.display import Image, display

COLORS = {"spark": "#E25A1C", "flink": "#4B9CD3"}

def _engines():
    return [e for e in ("spark", "flink") if e in results]

### 8.1 Throughput

In [ ]:
engines = _engines()
if engines:
    fig, ax = plt.subplots(figsize=(7, 4))
    vals  = [results[e].get("throughput_rps", 0) for e in engines]
    bars  = ax.bar(engines, vals, color=[COLORS[e] for e in engines],
                   width=0.45, edgecolor="white", linewidth=1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f"{v:,.0f}", ha="center", va="bottom", fontsize=11)
    ax.set_title("Throughput — Spark vs Flink", fontsize=13, fontweight="bold")
    ax.set_ylabel("Registros / segundo")
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_xticklabels([e.capitalize() for e in engines], fontsize=12)
    ax.grid(axis="y", alpha=0.35)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    path = PLOTS_DIR / "throughput.png"
    plt.savefig(path, dpi=150)
    plt.close()
    display(Image(path))

### 8.2 Latência por percentil

In [ ]:
engines = _engines()
if engines:
    fig, ax = plt.subplots(figsize=(7, 4))
    percentiles = ["p50", "p95", "p99"]
    x = np.arange(len(percentiles))
    w = 0.35
    for i, e in enumerate(engines):
        vals = [results[e].get(f"latency_{p}", 0) for p in percentiles]
        offset = (i - len(engines)/2 + 0.5) * w
        rects = ax.bar(x + offset, vals, w, label=e.capitalize(),
                       color=COLORS[e], alpha=0.9, edgecolor="white")
        for r, v in zip(rects, vals):
            ax.text(r.get_x() + r.get_width()/2, r.get_height() + 1,
                    f"{v:.0f}", ha="center", va="bottom", fontsize=9)
    ax.set_title("Latência por Percentil", fontsize=13, fontweight="bold")
    ax.set_ylabel("Latência (ms)")
    ax.set_xticks(x); ax.set_xticklabels(["P50", "P95", "P99"], fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.35)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    path = PLOTS_DIR / "latency_percentiles.png"
    plt.savefig(path, dpi=150)
    plt.close()
    display(Image(path))

### 8.3 CDF da latência

In [ ]:
engines = _engines()
if engines:
    fig, ax = plt.subplots(figsize=(8, 4))
    for e in engines:
        lats = sorted(results[e].get("latency_ms_raw", []))
        if lats:
            cdf = np.arange(1, len(lats)+1) / len(lats)
            ax.plot(lats, cdf, label=e.capitalize(), color=COLORS[e], linewidth=2)
    for p, ls in [(0.50,"--"),(0.95,":"),(0.99,"-.")]:
        ax.axhline(p, color="gray", linestyle=ls, linewidth=0.9, alpha=0.7)
        ax.text(ax.get_xlim()[1]*0.01 or 1, p+0.01, f"P{int(p*100)}",
                fontsize=8, color="gray")
    ax.set_title("CDF da Latência", fontsize=13, fontweight="bold")
    ax.set_xlabel("Latência (ms)")
    ax.set_ylabel("Probabilidade acumulada")
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    path = PLOTS_DIR / "latency_cdf.png"
    plt.savefig(path, dpi=150)
    plt.close()
    display(Image(path))

### 8.4 Distribuição de latência (histograma)

In [ ]:
engines = _engines()
if engines:
    fig, ax = plt.subplots(figsize=(8, 4))
    for e in engines:
        lats = results[e].get("latency_ms_raw", [])
        if lats:
            ax.hist(lats, bins=40, alpha=0.55, label=e.capitalize(),
                    color=COLORS[e], edgecolor="none", density=True)
    ax.set_title("Distribuição de Latência", fontsize=13, fontweight="bold")
    ax.set_xlabel("Latência (ms)")
    ax.set_ylabel("Densidade")
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    path = PLOTS_DIR / "latency_histogram.png"
    plt.savefig(path, dpi=150)
    plt.close()
    display(Image(path))

### 8.5 Throughput vs Tempo total (painel comparativo)

In [ ]:
engines = _engines()
if engines:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Spark vs Flink — Dataset 3W (Petrobras)",
                 fontsize=13, fontweight="bold")

    # --- Throughput + tempo (dual axis) ---
    ax, ax2 = axes[0], axes[0].twinx()
    x = np.arange(len(engines))
    w = 0.35
    throughputs = [results[e].get("throughput_rps", 0) for e in engines]
    times       = [results[e].get("total_time_s", 0)   for e in engines]
    bars1 = ax.bar(x - w/2, throughputs, w, label="Throughput (rec/s)",
                   color=[COLORS[e] for e in engines], alpha=0.88, edgecolor="white")
    bars2 = ax2.bar(x + w/2, times, w, label="Tempo total (s)",
                    color=[COLORS[e] for e in engines], alpha=0.4,
                    edgecolor="white", hatch="///")
    for bar, v in zip(bars1, throughputs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(throughputs)*0.01,
                f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
    for bar, v in zip(bars2, times):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(times)*0.01,
                 f"{v:.1f}s", ha="center", va="bottom", fontsize=9, style="italic")
    ax.set_title("Throughput e Tempo")
    ax.set_ylabel("Throughput (rec/s)")
    ax2.set_ylabel("Tempo (s)", color="gray")
    ax.set_xticks(x); ax.set_xticklabels([e.capitalize() for e in engines])
    ax.set_ylim(0, max(throughputs)*1.3 if throughputs else 1)
    ax2.set_ylim(0, max(times)*1.3 if times else 1)
    ax.grid(axis="y", alpha=0.3)
    p1 = mpatches.Patch(color="gray", alpha=0.88, label="Throughput (rec/s)")
    p2 = mpatches.Patch(color="gray", alpha=0.4, hatch="///", label="Tempo total (s)")
    ax.legend(handles=[p1, p2], fontsize=8, loc="upper right")

    # --- Latência P50/P95/P99 agrupada ---
    ax3 = axes[1]
    pcts = ["p50", "p95", "p99"]
    x2 = np.arange(len(pcts)); w2 = 0.35
    for i, e in enumerate(engines):
        vals = [results[e].get(f"latency_{p}", 0) for p in pcts]
        off = (i - len(engines)/2 + 0.5) * w2
        rs = ax3.bar(x2+off, vals, w2, label=e.capitalize(),
                     color=COLORS[e], alpha=0.9, edgecolor="white")
        for r, v in zip(rs, vals):
            ax3.text(r.get_x()+r.get_width()/2, r.get_height()+1,
                     f"{v:.0f}", ha="center", va="bottom", fontsize=9)
    ax3.set_title("Latência por Percentil")
    ax3.set_ylabel("Latência (ms)")
    ax3.set_xticks(x2); ax3.set_xticklabels(["P50","P95","P99"], fontsize=11)
    ax3.legend(fontsize=10); ax3.grid(axis="y", alpha=0.3)
    ax3.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    path = PLOTS_DIR / "benchmark_overview.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    display(Image(path))
    print(f"\nTodos os plots salvos em: {PLOTS_DIR}/")